In [1]:
# import functions and libraries
from src.load_data      import load_tile_local
from src.load_data      import load_trees_local
from src.convert_crs    import unify_crs
from src.convert_crs    import trees_geo_to_pixel
from src.tree_selection import tree_selection
from src.tree_selection import remove_non_trees
#from src.box_creation   import create_boxes
#from src.box_creation   import plot_boxes_on_tile
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Settings
data_path   = Path("../DATA")
ntiles      = 150
file_ext    = ".jp2"
target_tile = 0
#box_width   = 160;
#output_path = f"tile_with_boxes_boxwidth{box_width}.png"
filtered_tree_path = ("../DATA/baeume.gpkg")

In [3]:
# load aerial photograph(s) from local storage
tiles = load_tile_local(data_path,ntiles,file_ext)

# print(tiles)

In [4]:
# load tree coordinates from local storage
trees = load_trees_local(data_path)

#trees.head(10)

In [5]:
# convert crs of trees to the same epsg as in the aerial/tile data
trees = unify_crs(tiles,trees)

#trees.head(10)

In [6]:
# remove entries that are no real trees
trees = remove_non_trees(filtered_tree_path, trees)

#trees.head(10)

In [7]:
trees.head(10)

,str_schl,baumgruppe,geometry,isTrue
0,02505,Tilia,POINT (404685.243 5759126.262),True
1,02505,Tilia,POINT (404698.459 5759130.288),True
2,02505,Carpinus,POINT (404692.992 5759278.839),True
3,02505,Carpinus,POINT (404692.909 5759289.147),True
4,02505,Carpinus,POINT (404692.118 5759319.032),True
5,02505,Tilia,POINT (404685.647 5759520.435),True
6,02505,Tilia,POINT (404683.583 5759605.462),True
8,02505,Carpinus,POINT (404662.299 5760349.830),True
9,06350,Platanus,POINT (404352.732 5758911.890),True
10,06350,Platanus,POINT (404318.559 5758966.392),True


In [8]:
# Übersicht aller Baumarten, die in den Katasterdaten vorkommen.
species = sorted(trees["baumgruppe"].unique())

print(species)

['Abies', 'Acer', 'Aesculus', 'Ailanthus', 'Alnus', 'Amelanchier', 'Betula', 'Carpinus', 'Castanea', 'Catalpa', 'Cedrus', 'Celtis', 'Cercidiphyllum', 'Chamaecyparis', 'Cornus', 'Corylus', 'Crataegus', 'Cryptomeria', 'Davidia', 'Decaisnea', 'Euonymus', 'Fagus', 'Fraxinus', 'Ginkgo', 'Gleditsia', 'Ilex', 'Juglans', 'Juniperus', 'Koelreuteria', 'Laburnum', 'Larix', 'Liquidambar', 'Liriodendron', 'Magnolia', 'Malus', 'Malus-Hybride', 'Mespilus', 'Metasequoia', 'Metasequoia glyptostroboides', 'Nyssa', 'Parrotia', 'Paulownia', 'Picea', 'Pinus', 'Platanus', 'Populus', 'Prunus', 'Pseudotsuga', 'Pterocarya', 'Pyrus', 'Quercus', 'Robinia', 'Salix', 'Sambucus', 'Sequoiadendron', 'Sophora', 'Sorbus', 'Taxodium', 'Taxus', 'Thuja', 'Tilia', 'Tsuga', 'Ulmus', 'Viburnum', 'Zelkova']


In [9]:
print(trees["baumgruppe"].value_counts())

Tilia          10279
Quercus         8199
Acer            5324
Carpinus        3448
Platanus        1965
               ...  
Sambucus           1
Davidia            1
Mespilus           1
Cryptomeria        1
Abies              1
Name: baumgruppe, Length: 65, dtype: int64


In [10]:
summary = pd.DataFrame(
    columns=["tile", "gesamt"] + species
)



# Alle Spalten definieren
columns = ["tile", "gesamt"] + species

# Leeren DataFrame anlegen
summary = pd.DataFrame(columns=columns)


for tile in range(len(tiles)):

    #print(tile, tiles[tile]["name"])
    
    # convert tree coordinates to pixel values
    # function adds it to the geopandas dataframe
    trees_pix = trees_geo_to_pixel(tiles, trees, tile)
    
    # the trees dataframe contains trees of all Münster
    # select those that are within the tile. 
    trees_within_tile = tree_selection(tiles, trees_pix, tile)
    counts = trees_within_tile["baumgruppe"].value_counts()
    
    #print(tile, len(trees_within_tile))
    
    # Eine Zeile als Dictionary erzeugen
    row = {
        "tile": tiles[tile]["name"],
        "gesamt": len(trees_within_tile)
    }

    # Für jede Baumart Anzahl eintragen
    for s in species:
        row[s] = counts.get(s, 0)

    # Zeile anhängen
    summary.loc[len(summary)] = row
    
    

In [11]:
# Summary ist ein Dataframe mit einer Zeile pro Kachel. 
# In jeder Zeile steht wie viele Bäume von welcher Art als auch insgesamt vorkommen.
summary.head(150)

,tile,gesamt,Abies,Acer,Aesculus,Ailanthus,Alnus,Amelanchier,Betula,Carpinus,...,Sophora,Sorbus,Taxodium,Taxus,Thuja,Tilia,Tsuga,Ulmus,Viburnum,Zelkova
0,dop10rgbi_32_400_5759_1_nw_2020.jp2,112,0,28,0,0,0,0,1,8,...,0,0,0,0,0,3,0,0,0,0
1,dop10rgbi_32_404_5751_1_nw_2020.jp2,3,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,dop10rgbi_32_403_5749_1_nw_2020.jp2,94,0,10,4,0,0,0,0,0,...,0,0,0,0,0,65,0,0,0,0
3,dop10rgbi_32_409_5751_1_nw_2020.jp2,14,0,1,0,0,0,0,0,3,...,0,2,0,0,0,0,0,0,0,0
4,dop10rgbi_32_407_5756_1_nw_2020.jp2,192,0,28,0,0,0,0,5,25,...,0,8,0,0,0,68,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,dop10rgbi_32_407_5757_1_nw_2020.jp2,694,0,152,28,2,6,0,7,20,...,1,26,0,1,0,272,0,0,0,0
146,dop10rgbi_32_401_5760_1_nw_2020.jp2,44,0,6,0,0,0,2,6,6,...,0,0,0,0,0,2,0,0,0,0
147,dop10rgbi_32_400_5758_1_nw_2020.jp2,5,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
148,dop10rgbi_32_404_5750_1_nw_2020.jp2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# Aus den Kachelnamen kann man die geografische Position der Kacheln ablesen.
summary["x"] = summary["tile"].str.extract(r'_(\d{3})_').astype(int)
summary["y"] = summary["tile"].str.extract(r'_(\d{4})_').astype(int)

In [13]:
# Speichern
summary.to_csv("df_trees_per_tile.csv")

In [ ]:
# erstelle eine heatmap für die Gesamtanzahl der Bäume
heatmap = summary.pivot(
    index="y",
    columns="x",
    values="gesamt"
)
heatmap = heatmap.astype(float)

In [ ]:
# heatmap plotten
plt.figure(figsize=(10,6))
plt.imshow(heatmap, origin="upper")
plt.colorbar(label="Anzahl Bäume")
plt.xlabel("West → Ost")
plt.ylabel("Nord → Süd")
plt.show()

In [ ]:
# noch einmal numerisch
print(heatmap)

In [ ]:
# berechne die kumulative Summe über die Spalten (wichtig für die Aufteilung)

variables = ["gesamt"] + species

result = pd.DataFrame(columns=sorted(summary["x"].unique()))

for var in variables:

    heatmap = summary.pivot(
        index="y",
        columns="x",
        values=var
    )

    column_sum = heatmap.sum(axis=0)
    cum_sum = column_sum.cumsum()

    result.loc[var] = cum_sum
    
    
result_pct = result.div(result.iloc[:, -1], axis=0)

In [ ]:
# numnerische Ausgabe
print(result_pct)

In [ ]:
# Split abgeleitet von der Gesamtzahl:
# Wie viel % jeder Baumart sind dann inn den jeweiligen Datensätzen vorhanden?
# Spalte 400-405: Trainingsdaten
# Spalte 406: Validierung
# Spalte 407-409: Test
‚
variables = ["gesamt"] + species

split_summary = pd.DataFrame(
    columns=["Training", "Validation", "Test"],
    index=variables
)

for var in variables:

    # Summe pro West-Ost-Spalte
    column_sum = (
        summary
        .groupby("x")[var]
        .sum()
    )

    total = column_sum.sum()

    train = column_sum.loc[400:405].sum()
    valid = column_sum.loc[406]
    test  = column_sum.loc[407:409].sum()

    split_summary.loc[var] = [
        100 * train / total,
        100 * valid / total,
        100 * test / total
    ]
    
split_summary = split_summary.astype(float).round(1)
split_summary.head(25)

In [ ]:
# speichern
result_pct.to_csv("df_kumulativeSummeBaumartenUeberSpalten.csv")
split_summary.to_csv("df_PronzentBaumartProDatensatz.csv")